In [2]:
import os
import torch
from collections import Counter
from torch.utils.data import WeightedRandomSampler

import kagglehub
path = kagglehub.dataset_download("aklimarimi/8-facial-expressions-for-yolo")

base = f"{path}/9 Facial Expressions you need"
classes = ['angry', 'contempt', 'disgust', 'fear', 'happy', 'natural', 'sad', 'sleepy', 'surprised']

In [3]:
import torch
import os
from torch.utils.data import WeightedRandomSampler

# Count class frequencies from train labels
label_dir = f"{base}/train/labels"
class_counts_train = Counter()

for lf in os.listdir(label_dir):
    with open(f"{label_dir}/{lf}") as f:
        lines = f.readlines()
    if lines:
        cls_id = int(lines[0].strip().split()[0])
        class_counts_train[cls_id] += 1

print("Train class counts:", dict(sorted(class_counts_train.items())))

Train class counts: {0: 11195, 1: 2543, 2: 4296, 3: 5358, 4: 13831, 5: 5663, 6: 11969, 7: 1051, 8: 8960}


In [4]:
# ── Class weights for loss function ───────────────────────────────────────────
total = sum(class_counts_train.values())
class_weights = torch.tensor([
    total / class_counts_train[i] for i in range(len(classes))
], dtype=torch.float)

print("Class weights:")
for i, (cls, w) in enumerate(zip(classes, class_weights)):
    print(f"  {cls}: {w:.4f}")

# ── Sample weights for WeightedRandomSampler ──────────────────────────────────
# one weight per image in the training set
label_dir = f"{base}/train/labels"
sample_weights = []

for lf in sorted(os.listdir(label_dir)):
    with open(f"{label_dir}/{lf}") as f:
        lines = f.readlines()
    if lines:
        cls_id = int(lines[0].strip().split()[0])
        sample_weights.append(class_weights[cls_id].item())
    else:
        sample_weights.append(1.0)  # fallback for empty label files

sample_weights = torch.tensor(sample_weights, dtype=torch.float)

# ── Sampler ───────────────────────────────────────────────────────────────────
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"\nSampler ready — {len(sample_weights)} samples")
print(f"Class weights tensor: {class_weights}")

Class weights:
  angry: 5.7942
  contempt: 25.5077
  disgust: 15.0992
  fear: 12.1064
  happy: 4.6899
  natural: 11.4544
  sad: 5.4195
  sleepy: 61.7184
  surprised: 7.2395

Sampler ready — 64866 samples
Class weights tensor: tensor([ 5.7942, 25.5077, 15.0992, 12.1064,  4.6899, 11.4544,  5.4195, 61.7184,
         7.2395])


In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

class FacialExpressionDataset(Dataset):
    def __init__(self, base, split, transform=None):
        self.img_dir   = f"{base}/{split}/images"
        self.label_dir = f"{base}/{split}/labels"
        self.transform = transform
        self.samples   = []  # list of (img_path, class_id)

        for lf in sorted(os.listdir(self.label_dir)):
            stem = os.path.splitext(lf)[0]
            label_path = f"{self.label_dir}/{lf}"

            # find the image file
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                candidate = f"{self.img_dir}/{stem}{ext}"
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            if img_path is None:
                continue  # skip if no matching image

            # read class id from first line of label file
            with open(label_path) as f:
                lines = f.readlines()
            if not lines:
                continue  # skip empty label files

            cls_id = int(lines[0].strip().split()[0])
            self.samples.append((img_path, cls_id))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, cls_id = self.samples[idx]
        img = Image.open(img_path).convert("RGB")  # handles grayscale too
        if self.transform:
            img = self.transform(img)
        return img, cls_id


# Quick sanity check
dataset = FacialExpressionDataset(base, "train")
print(f"Dataset size: {len(dataset)}")
img, label = dataset[0]
print(f"Sample — label: {classes[label]}, image size: {img.size}")